### Experiment with GPT4o

In [ ]:
import pandas as pd
import libs.prompts as prompts
from libs.exp_utils import classify_dataset, evaluate_experiment, call_openai

import json

In [ ]:
# Define model name
MODEL_NAME = "gpt-4o"

### Dataset load

In [ ]:
# Load the dataset and convert to appropriate data types
csv_path = "../../Datasets/dataset.csv"  # Path to the CSV file
df = pd.read_csv(csv_path)

# Ensure that data types are correct
df['has_error'] = df['has_error'].astype(bool)

subset = df.reset_index(drop=True)

### Experiment With Combined Dataset

#### Zero-shot Prompting

In [ ]:
results_df = classify_dataset(subset, prompts.create_zeroshot_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)

# Display the metrics
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")

#### Zero-shot CoT Prompting

In [ ]:
results_df = classify_dataset(subset, prompts.create_zeroshot_cot_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)

# Display the metrics
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")


#### Zero-shot ToT Prompting

In [ ]:
results_df = classify_dataset(subset, prompts.create_zeroshot_tot_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)

# Display the metrics
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")

#### Zero-shot ToT Prompting - Testing False Positives and False Negatives

In [ ]:
# 1. Classification
results_df = classify_dataset(subset, prompts.create_zeroshot_tot_prompt, MODEL_NAME)

# 2. Evaluation
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")

# 3. Identify False Positives and False Negatives
fp = results_df[
    (results_df['predicted_has_error'] == True) & 
    (results_df['actual_has_error'] == False)
]

fn = results_df[
    (results_df['predicted_has_error'] == False) & 
    (results_df['actual_has_error'] == True)
]

# 4. Display FALSE POSITIVE
if not fp.empty:
    fp_idx = fp.index[0]
    file_path = subset.loc[fp_idx, 'file_path']
    prompt_txt = prompts.create_zeroshot_tot_prompt(f"../../Datasets/{file_path}")
    response = call_openai(MODEL_NAME, prompt_txt)

    print("\n🔴 FALSE POSITIVE:")
    print("🔹 Prompt:\n", prompt_txt)
    print("🔹 LLM Response:\n", response)
    print("🔹 Ground Truth (should be False):", results_df.loc[fp_idx, 'actual_has_error'])
    print("🔹 Model Prediction (was True):", results_df.loc[fp_idx, 'predicted_has_error'])
else:
    print("No false positives found.")

# 5. Display FALSE NEGATIVE
if not fn.empty:
    fn_idx = fn.index[0]
    file_path = subset.loc[fn_idx, 'file_path']
    prompt_txt = prompts.create_zeroshot_tot_prompt(f"../../Datasets/{file_path}")
    response = call_openai(MODEL_NAME, prompt_txt)

    print("\n🔵 FALSE NEGATIVE:")
    print("🔹 Prompt:\n", prompt_txt)
    print("🔹 LLM Response:\n", response)
    print("🔹 Ground Truth (should be True):", results_df.loc[fn_idx, 'actual_has_error'])
    print("🔹 Model Prediction (was False):", results_df.loc[fn_idx, 'predicted_has_error'])
else:
    print("No false negatives found.")
